In [1]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import tqdm
from language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from language_models.model import RNNModel as lstm
from language_models.utils import move_to_device
import random
import pandas as pd
from collections import defaultdict
import numpy as np


In [35]:
batch_size = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)
checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/epoch_40.pt"  # Replace with your checkpoint path


In [7]:
model = lstm('LSTM', 50001, 650, 650, 2, 0.2, False)
with open(checkpoint_path, 'rb') as f:
    state_dict = torch.load(f, map_location='cuda' if device =='cuda' else 'cpu')
    model.load_state_dict(state_dict['model_state_dict'])
model.to(device)
model.eval() 

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 650)
  (rnn): LSTM(650, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

In [8]:
simple = '/scratch2/mrenaudin/colorlessgreenRNNs/simple.txt'

In [9]:
from collections import defaultdict

class SimplePairDataset(Dataset):
    """
    Dataset that reads the simple file format:
    Sentence \t condition \t label(correct/wrong) \t id
    Groups pairs by id: for each id, stores (correct_sentence, wrong_sentence, condition)
    """
    def __init__(self, filepath, dictionary):
        self.dictionary = dictionary
        self.pairs = []  # list of dicts with keys: 'correct_sent', 'wrong_sent', 'condition'

        # Temp storage for grouping by id
        data_by_id = defaultdict(dict)

        with open(filepath, "r") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 4:
                    continue  # skip malformed lines
                sentence_str, condition, label, pair_id = parts
                # Tokenize the sentence (split by spaces)
                sentence_tokens = sentence_str.split()

                # Encode sentence words to indices, use <unk> if not found
                encoded_sentence = [
                    self.dictionary.word2idx.get(w, self.dictionary.word2idx.get("<unk>"))
                    for w in sentence_tokens
                ]

                data_by_id[pair_id][label] = {
                    "sentence": sentence_tokens,
                    "encoded_sentence": torch.tensor(encoded_sentence, dtype=torch.long),
                }
                data_by_id[pair_id]["condition"] = condition

        # Now create list of pairs
        for pair_id, val in data_by_id.items():
            if "correct" in val and "wrong" in val:
                self.pairs.append({
                    "correct_sentence": val["correct"]["sentence"],
                    "encoded_correct_sentence": val["correct"]["encoded_sentence"],
                    "wrong_sentence": val["wrong"]["sentence"],
                    "encoded_wrong_sentence": val["wrong"]["encoded_sentence"],
                    "condition": val["condition"],
                })

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        return {
            "correct_sentence": pair["correct_sentence"],
            "encoded_correct_sentence": pair["encoded_correct_sentence"],
            "wrong_sentence": pair["wrong_sentence"],
            "encoded_wrong_sentence": pair["encoded_wrong_sentence"],
            "condition": pair["condition"],
        }


In [11]:
test_dataset = SimplePairDataset(simple, dictionary)

In [37]:
def collate_fn_pairs(batch):
    # batch is list of dicts as returned by __getitem__
    return {
        "correct_sentence": [item["correct_sentence"] for item in batch],
        "encoded_correct_sentence": torch.nn.utils.rnn.pad_sequence(
            [item["encoded_correct_sentence"] for item in batch], batch_first=True
        ),
        "wrong_sentence": [item["wrong_sentence"] for item in batch],
        "encoded_wrong_sentence": torch.nn.utils.rnn.pad_sequence(
            [item["encoded_wrong_sentence"] for item in batch], batch_first=True
        ),
        "condition": [item["condition"] for item in batch],
    }


In [38]:
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn_pairs)

In [39]:
init_sentence = " ".join(["In service , the aircraft was operated by a crew of five and could accommodate either 30 paratroopers , 32 <unk> and 28 sitting casualties , or 50 fully equipped troops . <eos>",
                    "He even speculated that technical classes might some day be held \" for the better training of workmen in their several crafts and industries . <eos>",
                    "After the War of the Holy League in 1537 against the Ottoman Empire , a truce between Venice and the Ottomans was created in 1539 . <eos>",
                    "Moore says : \" Tony and I had a good <unk> and off-screen relationship , we are two very different people , but we did share a sense of humour \" . <eos>",
                    "<unk> is also the basis for online games sold through licensed lotteries . <eos>"])

In [40]:
def feed_input(model, hidden, w):
    inp = torch.autograd.Variable(torch.LongTensor([[dictionary.word2idx[w]]]))
    out, hidden = model(inp, hidden)
    return out, hidden
def feed_sentence(model, h, sentence):
    outs = []
    for w in sentence:
        out, h = feed_input(model, h, w)
        outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))
    return outs, h

In [41]:
model.eval()
hidden = model.init_hidden(1) 
init_out, init_h = feed_sentence(model, hidden, init_sentence.split(" "))



/tmp/ipykernel_17460/915584119.py:9: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))


In [50]:
def get_last_word_log_prob(encoded_sentences, hidden):
    out = None
    for t in range(encoded_sentences.size(1) - 1):
        inp = encoded_sentences[:, t].unsqueeze(0).to(device)
        out, hidden = model(inp, hidden)
    log_probs = torch.nn.functional.log_softmax(out, dim=-1)
    # Target word is the last word in sentence
    targets = encoded_sentences[:, -1].to(device)
    # Gather log-prob of target word
    last_word_log_prob = log_probs[0, torch.arange(encoded_sentences.shape[0]), targets]
    return last_word_log_prob.cpu()

In [54]:
def eval_pairs(model, dataloader, init_sentence):
    condition_accuracies = defaultdict(int)
    condition_counts = defaultdict(int)
    sentence_details = []

    model.eval()
    hidden = model.init_hidden(1)
    _, init_h = feed_sentence(model, hidden, init_sentence.split(" "))

    with torch.no_grad():
        for batch in dataloader:
            batch_size = len(batch["condition"])

            # Expand init hidden states for batch
            hidden = (
                init_h[0].expand(-1, batch_size, -1).contiguous(),
                init_h[1].expand(-1, batch_size, -1).contiguous(),
            )

            # Evaluate each sentence in the batch: get log-prob of last wor
            correct_log_probs = get_last_word_log_prob(batch["encoded_correct_sentence"], hidden)
            wrong_log_probs = get_last_word_log_prob(batch["encoded_wrong_sentence"], hidden)

            preds = correct_log_probs >= wrong_log_probs

            for i in range(batch_size):
                cond = batch["condition"][i]
                pred = preds[i].item()
                condition_counts[cond] += 1
                condition_accuracies[cond] += pred
                sentence_details.append({
                    "correct_sentence": batch["correct_sentence"][i],
                    "wrong_sentence": batch["wrong_sentence"][i],
                    "condition": cond,
                    "correct_log_prob": correct_log_probs[i].item(),
                    "wrong_log_prob": wrong_log_probs[i].item(),
                    "model_prefers_correct": pred,
                })

    final_accuracies = {cond: condition_accuracies[cond] / condition_counts[cond]
                       for cond in condition_accuracies}
    return final_accuracies, sentence_details


In [55]:
acc = eval_pairs(model, test_dataloader, init_sentence)

/tmp/ipykernel_17460/915584119.py:9: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))


In [56]:
acc[0]

{'singular': 0.98, 'plural': 0.9966666666666667}